In [35]:
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

## Tools

In [7]:
from langchain.tools import tool

In [9]:
@tool
def tool_duckduckgo_search(query:str) -> str:
     """ Use this tool when you need to answer questions about current events or general knowledge. """
     from langchain_community.tools import DuckDuckGoSearchRun

     search = DuckDuckGoSearchRun()

     response = search.invoke(query)

     return response

In [10]:
tool_duckduckgo_search.invoke("what is the capital of france?")

'... of these cities is the capital, keep reading to find out. ... What is the capital city of France? The capital, and largest, city of France is Paris. Besides Paris, what is the capital of France? ... You use it between your head and your toes, the more it works the thinner it grows. What is the Capital of France? Paris ... Paris, the capital city of France, is one of the most famous and influential cities in the world. What is the Capital of France? ... As the capital city of France, the city plays host to the national government of France. However, Paris only became the official capital of France during the reign of Clovis I, in the late 5th and early 6th century.'

In [11]:
@tool
def tool_wikipedia_search(query:str) -> str:
    """ Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)

    return response

In [12]:
tool_wikipedia_search.invoke("Sam Altman")

'Page: Sam Altman\nSummary: Samuel Harris Altman (born April 22, 1985) is an American entrepreneur who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019.\nAltman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone geosocial networking service. Loopt was acquired by Green Dot Corporation for $43.4 million. In 2011, Altman joined Y Combinator, a technology startup accelerator and venture capital firm, and was the company\'s president from 2014 to 2019. He is a billionaire with many investments including Reddit, Worldcoin, and Helion Energy.\nAltman co-founded OpenAI in 2015 and became its CEO in 2019, a role that made him a prominent figure of the AI boom. He supervised the launch of ChatGPT in November 2022. In 2023, he was ousted by the organization\'s board of directors for not being "consistently candid". The move was met with significant backlash from employees and investors, resulting 

In [33]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics.

    Implementation notes:
    - Prefer the direct `arxiv` package (returns entries via `search.results()`).
    - If `arxiv` is not installed, fall back to the langchain_community wrapper/tool.
    - Return a formatted string (titles, urls, summaries) rather than printing.
    """

    try:
        import arxiv
    except Exception:
        # Fallback to langchain_community tool if `arxiv` package isn't available
        try:
            from langchain_community.tools import ArxivQueryRun
            from langchain_community.utilities import ArxivAPIWrapper

            arxiv_wrapper = ArxivAPIWrapper(
                top_k_results=3,
                doc_content_chars_max=2000,
            )
            arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)
            if hasattr(arxiv_tool, "invoke"):
                return arxiv_tool.invoke(query)
            return arxiv_tool.run(query)
        except Exception as e:
            return f"arXiv wrapper error: {e}"

    # Use arxiv package directly using the Client API (arxiv v4+)
    try:
        search = arxiv.Search(query=query, max_results=3)
        client = arxiv.Client()

        entries = None
        try:
            entries = list(client.results(search))
        except Exception:
            entries = None

        # If client.results failed, try old-style fallbacks
        if entries is None:
            # Try Search.results() if present
            try:
                if hasattr(search, "results"):
                    entries = list(search.results())
            except Exception:
                entries = None

        if entries is None and hasattr(arxiv, "query"):
            try:
                entries = arxiv.query(query=query, max_results=3)
            except Exception:
                entries = None

        if not entries:
            return "No results found."

        parts = []
        for e in entries:
            if isinstance(e, dict):
                title = e.get("title", "")
                summary = e.get("summary", "").strip().replace("\n", " ")
                url = e.get("id", e.get("entry_id", ""))
            else:
                title = getattr(e, "title", "")
                summary = getattr(e, "summary", "").strip().replace("\n", " ")
                url = getattr(e, "entry_id", getattr(e, "id", ""))

            parts.append(f"Title: {title}\nURL: {url}\nSummary: {summary}\n")

        return "\n\n".join(parts)
    except Exception as exc:
        return f"arxiv package error: {exc}"

In [34]:
tool_arxiv_search.invoke("What are the latest research papers about deep learning for natural language processing?")

"Title: Modular Mechanistic Networks: On Bridging Mechanistic and Phenomenological Models with Deep Neural Networks in Natural Language Processing\nURL: http://arxiv.org/abs/1807.09844v2\nSummary: Natural language processing (NLP) can be done using either top-down (theory driven) and bottom-up (data driven) approaches, which we call mechanistic and phenomenological respectively. The approaches are frequently considered to stand in opposition to each other. Examining some recent approaches in deep learning we argue that deep neural networks incorporate both perspectives and, furthermore, that leveraging this aspect of deep learning may help in solving complex problems within language technology, such as modelling language and perception in the domain of spatial cognition.\n\n\nTitle: The Modern Mathematics of Deep Learning\nURL: http://arxiv.org/abs/2105.04026v2\nSummary: We describe the new field of mathematical analysis of deep learning. This field emerged around a list of research qu

In [36]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "John Doe",
        "age": 30,
        "occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [37]:
tool_personal_info.invoke("John Doe")

'John Doe is 30 years old and works as a Software Engineer.'

## Bind Tools

In [39]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info
]

In [40]:
llm_bind = llm.bind_tools(toolkit)

In [41]:
llm_bind.invoke("what is the age of John Doe?")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 336, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DixhRspi6zoJ0KzubvZQND6cT3l8a', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e58eb-6bb6-77f2-a80b-725c117812d3-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_MjGNZcM4AMrIKQO0xPGv6TYD', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 336, 'output_tokens': 26, 'total_tokens': 362, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})